# 03 - Facade mean-Cp snapshots

Per-triangle mean-Cp renders per facade via `cfdmod.mesh_field`. Shown inline
AND written to `deliverables/<version>/facade/<body>/<direction>/`.

In [ ]:
import json, pathlib
import numpy as np
import matplotlib.pyplot as plt
import h5py
%matplotlib inline

from cfdmod import plot_config, mesh_field
from cfdmod.report import DebugWriter
from cfdmod.adapters.xdmf_h5 import XdmfH5Storage
from cfdmod.adapters.memory import MemoryFieldStore
from cfdmod.core import PointsDataSource, Topology, ElementMeta
from cfdmod.building import BuildingCase, cp_from_pressure
from ruamel.yaml import YAML

import logging
logging.getLogger("cfdmod").setLevel(logging.WARNING)
plot_config.apply_style()
yaml = YAML(typ="safe")
VERSION = "v3"

## Case config (read inline)

In [ ]:
project = pathlib.Path("/data/eng/consulting/NNN_CaseName")
CASE = "CaseName"  # results/<batch>/<CASE>_<dir>/... and <CASE>_noBody_<dir>/...
case_data = project / "post_processing/pp_config/case_data"
results   = project / "results"
dct_case = json.loads((case_data / "global_data.json").read_text())
from ruamel.yaml import YAML
yaml = YAML(typ="safe")

batch = dct_case["analysis"]["batch_name"]
H  = float(dct_case["H"]); L = float(dct_case["L"]); V0 = float(dct_case["V0"])
cats = sorted(k.split("directions_cat")[1] for k in dct_case["analysis"] if k.startswith("directions_cat"))
dirs_by_cat = {c: [str(d) for d in dct_case["analysis"][f"directions_cat{c}"]] for c in cats}
params_by_cat = {c: yaml.load((case_data / f"params_cat{c}.yaml").open()) for c in cats}

# bodies: prefer the per-body force_coefficient keys (building_<body>) since that
# is where the per-body floor divisions live; fall back to global_data body_names
fc_keys = list(params_by_cat[cats[0]]["force_coefficient"])
if len(fc_keys) > 1 and all(k.startswith("building_") for k in fc_keys):
    bodies = [k[len("building_"):] for k in fc_keys]
else:
    bodies = list(dct_case["analysis"]["body_names"])

simul_u_h, floor_heights = {}, {}
for c in cats:
    params = params_by_cat[c]
    cp0 = next(iter(params["pressure_coefficient"].values()))
    simul_u_h[c] = float(cp0["simul_U_H"]); fluid_density = float(cp0.get("fluid_density", 1.225))
    fc0 = next(iter(params["force_coefficient"].values()))
    nominal_area = float(fc0["nominal_area"])
    nominal_volume = float(next(iter(params["moment_coefficient"].values()))["nominal_volume"])
    if not floor_heights:
        for b in bodies:
            blk = params["force_coefficient"].get(f"building_{b}") or fc0
            floor_heights[b] = [float(z) for z in blk["bodies"][0]["sub_bodies"]["z_intervals"]]
directions = [d for c in cats for d in dirs_by_cat[c]]
category_of = lambda d: next(c for c, ds in dirs_by_cat.items() if d in ds)
print("bodies", bodies, "| directions", len(directions), "| simul_u_h", simul_u_h)

## Pick body + direction, read the body pressure

In [ ]:
body = next((b for b in bodies if b.lower() != "base"), bodies[0])
# body = "base"
wind_dir = directions[0]
mesh_path = project / f"run/artifacts/lnas/{body}.lnas"
folder = results / batch / f"{CASE}_{wind_dir}" / "000/probes/hist_series" / f"cp_analysis_{body}"
dbg = DebugWriter(pathlib.Path.cwd(), stage=f"facade/{body}", version=VERSION)

def read_point_probe(h5_path, time_axis):
    """Reference-pressure probe (points.point0.h5) -> PointsDataSource."""
    with h5py.File(h5_path, "r") as h:
        grp = h["pressure"]; keys = sorted(grp.keys(), key=lambda k: float(k[1:]))
        arr = np.empty((1, len(keys)), dtype=np.float32)
        for i, k in enumerate(keys):
            arr[0, i] = grp[k][()][0]
        pts = np.zeros((1, 3))
        if "Geometry" in h: pts[0] = np.asarray(h["Geometry"])[0]
    return PointsDataSource(time=time_axis, topology=Topology.points(pts),
                            elements=ElementMeta(position=pts), fields=MemoryFieldStore({"pressure": arr}))

cat = category_of(wind_dir)
storage = XdmfH5Storage(folder)
bdy  = storage.read_data_source(pathlib.Path(f"bodies.{body}"))
pref = read_point_probe(folder / "points.point0.h5", bdy.time)
case = BuildingCase(name=body, reference_height=H, characteristic_length=L,
                    basic_wind_speed=V0, simul_reference_velocity=simul_u_h[cat],
                    nominal_area=nominal_area, nominal_volume=nominal_volume,
                    floor_heights=floor_heights[body], fluid_density=fluid_density)

cp_mean = cp_from_pressure(bdy, pref, case, statistics=["mean"]).fields.read("mean")
print(f"{body} {wind_dir}: mean Cp range [{np.nanmin(cp_mean):.2f}, {np.nanmax(cp_mean):.2f}]")

## Render: iso view + per facade

In [ ]:
geom = mesh_field.load_geometry(str(mesh_path))
fig, _ = mesh_field.triangle_field_figure(geom, cp_mean, view=mesh_field.STANDARD_VIEWS["iso"],
                                          title=rf"{body} {float(wind_dir):.0f}$^\circ$ mean Cp", cbar_label="Cp [-]")
dbg.savefig(fig, f"{wind_dir}/cp_mean_iso.png", deliverable=True); plt.show()

for name, grp in mesh_field.facade_groups(str(mesh_path)).items():
    fig, _ = mesh_field.triangle_field_figure(geom, cp_mean, subset=grp,
                                              title=f"{name} mean Cp", cbar_label="Cp [-]")
    dbg.savefig(fig, f"{wind_dir}/cp_mean_{name}.png", deliverable=True); plt.show()
print("deliverables ->", dbg.deliverables_dir)